In [1]:
1+1

2

In [5]:
import os
import cohere
from dotenv import load_dotenv
load_dotenv()  # Loads from .env

True

In [6]:
cohere_api_key=os.getenv("COHERE_TOKEN")

In [8]:
co = cohere.ClientV2(cohere_api_key)

In [11]:
messages = [
    {"role": "system", "content": "You are a friendly person"},
    {"role": "user", "content": "hello! my name is carlos!"},
]

In [12]:
output = co.chat( model="command-a-03-2025", 
                 messages= messages  )
output = output.message.content[0].text
output

"Hello Carlos! It's nice to meet you. How can I assist you today?"

In [13]:
messages = [
    {"role": "system", "content": "You are a friendly person"},
    {"role": "user", "content": "do you remember my name?"},
]

In [14]:
output = co.chat( model="command-a-03-2025", 
                 messages= messages  )
output = output.message.content[0].text
output

"As an AI, I don't have the ability to remember personal details like names unless they are provided in the current conversation. If you'd like, you can share your name, and I'll do my best to address you by it during our chat! 😊"

### Cohere LangChain

In [ ]:
#pip install langchain-cohere

In [ ]:
# https://github.com/googlecolab/colabtools/issues/5455
# For langchain-cohere==0.4.4 downgrade cohere to 5.15.0, it will solve the problem.

#%pip uninstall cohere
#%pip install cohere==5.15.0

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # Loads from .env
cohere_api_key=os.getenv("COHERE_TOKEN")

In [ ]:
from langchain_cohere import ChatCohere
from langchain_core.messages import AIMessage, HumanMessage

In [4]:
# Define the Cohere LLM
llm = ChatCohere(
    cohere_api_key=cohere_api_key, model="command-a-03-2025"
)

In [5]:
# Send a chat message without chat history
current_message = [HumanMessage(content="knock knock")]
print(llm.invoke(current_message))

content="Who's there?" additional_kwargs={'id': 'c1b7b9f9-e014-40e2-acaa-24a862c4bb05', 'finish_reason': 'COMPLETE', 'content': "Who's there?", 'token_count': {'input_tokens': 497.0, 'output_tokens': 7.0}} response_metadata={'id': 'c1b7b9f9-e014-40e2-acaa-24a862c4bb05', 'finish_reason': 'COMPLETE', 'content': "Who's there?", 'token_count': {'input_tokens': 497.0, 'output_tokens': 7.0}} id='run--beaea745-3e40-4855-ba31-e6d3c807268c-0' usage_metadata={'input_tokens': 497, 'output_tokens': 7, 'total_tokens': 504}


In [6]:
# Send a chat message with chat history, note the last message is the current user message
current_message_and_history = [
    HumanMessage(content="knock knock"),
    AIMessage(content="Who's there?"),
    HumanMessage(content="Tank"),
]
print(llm.invoke(current_message_and_history))

content='Tank who?' additional_kwargs={'id': '56b72955-1f3d-4f62-9715-e0bf8bef658a', 'finish_reason': 'COMPLETE', 'content': 'Tank who?', 'token_count': {'input_tokens': 510.0, 'output_tokens': 5.0}} response_metadata={'id': '56b72955-1f3d-4f62-9715-e0bf8bef658a', 'finish_reason': 'COMPLETE', 'content': 'Tank who?', 'token_count': {'input_tokens': 510.0, 'output_tokens': 5.0}} id='run--fc0f1abb-24b5-474b-9ebc-f9f2bb6dc36c-0' usage_metadata={'input_tokens': 510, 'output_tokens': 5, 'total_tokens': 515}


In [13]:
# Memory start
from langchain.chains import ConversationChain
from langchain.memory import ConversationSummaryMemory
from langchain import PromptTemplate
from langchain import LLMChain


In [14]:

# Create a summary prompt template
# I dont know if the tokens <s><|user|> are useful for this model
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""

summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)


# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

C:\Users\Carlos Ivan\AppData\Local\Temp\ipykernel_13752\2483041381.py:39: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


In [15]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Carlos. What is 1 + 4?"})


{'input_prompt': 'Hi! My name is Carlos. What is 1 + 4?',
 'chat_history': '',
 'text': 'Hi Carlos! Nice to meet you. The answer to 1 + 4 is **5**. How can I assist you further?'}

In [16]:
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': '**New summary:**\n\nThe conversation begins with Carlos introducing himself and asking a simple math question: "What is 1 + 4?" The AI responds by greeting Carlos, providing the correct answer (**5**), and offering further assistance.  \n\n**Updated summary:**\n\nThe conversation starts with Carlos introducing himself and asking a basic math question, "What is 1 + 4?" The AI greets Carlos, correctly answers **5**, and inquires how it can assist him further.',
 'text': 'Your name is **Carlos**.'}

In [17]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': '**New summary:**\n\nThe conversation begins with Carlos introducing himself and asking a simple math question: "What is 1 + 4?" The AI responds by greeting Carlos, providing the correct answer (**5**), and offering further assistance. Carlos then asks, "What is my name?" The AI correctly identifies his name as **Carlos**.  \n\n**Updated summary:**\n\nThe conversation starts with Carlos introducing himself and asking a basic math question, "What is 1 + 4?" The AI greets Carlos, correctly answers **5**, and inquires how it can assist him further. Carlos follows up by asking, "What is my name?" The AI accurately responds with his name, **Carlos**.',
 'text': 'The first question you asked was, "What is 1 + 4?"'}

In [18]:
# Check what the summary is thus far
memory.load_memory_variables({})

{'chat_history': '**New summary:**\n\nThe conversation begins with Carlos introducing himself and asking a basic math question, "What is 1 + 4?" The AI greets Carlos, correctly answers **5**, and inquires how it can assist him further. Carlos follows up by asking, "What is my name?" The AI accurately responds with his name, **Carlos**. Later, Carlos asks, "What was the first question I asked?" The AI correctly recalls and answers, "The first question you asked was, \'What is 1 + 4?\'"'}

In [ ]:
# Code from project sopra

In [ ]:

"""
predict-command-a.py

Predicts review sentiments on the test data using the generative model Command A from Cohere. Returns predictions in 
a pandas df with a format ready to benchmark, and other information of relevance to be analyzed such as inference time.

Usage:
    python predict-command-a.py 
"""

import pandas as pd
import os
import time
import json
from pathlib import Path


# Add scripts/utils/ to path, to load package functions
import sys
sys.path.append(str(Path(__file__).resolve().parent / "utils"))
from auxiliar_functions import load_test_data, save_outputs






def get_sentiment(reviews):
    """
    Calculates sentiment of a review or a list of reviews using the generative model Command A from Cohere.
    Due to rate limit issues, there is a gap of 6 seconds between calls to the model. 

    Args:
        reviews (str or list): a single review (str) or a list of reviews (list)

    Returns:
        df (pandas dataframe): dataframe containing reviews, their sentiment, probability of being positive, 
        and predicted sentiment.
    """

    # Make sure input is a list (output of hf is 3 classes if a string is given, or just the top class if a list is given)
    if not isinstance(reviews, list):
        reviews = [reviews]
    co = cohere.ClientV2(cohere_api_key)
    
    prompt="""Determine if the following document is a positive or negative movie review:
    [REVIEW]

    If it is positive, return 1, and if it is negative return 0. Do not give any other answers.
    """

    # Inference for all reviews
    all_rows = []
    for i, review in enumerate(reviews):
        review = review[:8000] # truncation for maximum context length

        messages = [
            {"role": "system", "content": "You are an expert in movie reviews"},
            {"role": "user", "content": prompt.replace("[REVIEW]", review)},
        ]

        output = co.chat( model="command-a-03-2025", messages= messages  )
        output = output.message.content[0].text
        print(i)
        time.sleep(6)

        all_rows.append({
            "review_index": i,
            "review" : review, 
            "positive_score": int(output)
        })

    # Create DataFrame
    df = pd.DataFrame(all_rows)

    outputs_list = ['positive' if score > 0.5 else 'negative' for score in df['positive_score']]
    df['Prediction']=outputs_list
    return df




model_name = 'generative-command-a'
adaptations = ''
other_comments = 'Rate limit of 10 requests per minute.'


if __name__ == "__main__":
    data = load_test_data()

    start = time.time()
    predictions = get_sentiment(list(data.review))
    end = time.time()
    inference_time = end - start

    save_outputs(data, predictions, model_name, adaptations, inference_time, other_comments)